# DAGonStar tutorial — Jupyter and Google Colab

This notebook is the executable companion to the fifteen lessons in `docs/tutorial/`. Run cells in order. It is designed for a fresh Google Colab runtime, but also works in a local Jupyter kernel with Python 3.8+.

**Scope:** local batch workflows are directly runnable. Docker, SSH, Slurm, and cloud backends need the infrastructure and credentials described in their lessons. Do not put credentials in this notebook.

In [ ]:
# Colab setup: clone the default branch and install this checkout.
!git clone --branch development https://github.com/DagOnStar/dagonstar.git
%cd dagonstar
!pip install -e .

In [ ]:
from pathlib import Path
import tempfile

from dagon import Workflow
from dagon.task import DagonTask, TaskType

def local_config(workdir):
    return {
        'batch': {'scratch_dir_base': str(workdir), 'run_base': '', 'remove_dir': 'False', 'threads': '1'},
        'dagon_service': {'use': 'False'},
        'ftp_pub': {'ip': 'localhost', 'user': 'anonymous', 'password': ''},
    }

def workspace(name):
    return Path(tempfile.mkdtemp(prefix=f'dagonstar-{name}-'))

print('DAGonStar import OK')

## Lesson 01 — Create a local workflow

A workflow owns tasks. A local batch task executes a shell command in its scratch directory. The following writes one file and prints its location.

In [ ]:
workdir = workspace('lesson01')
workflow = Workflow('Lesson01', config=local_config(workdir))
hello = DagonTask(TaskType.BATCH, 'hello', "echo 'hello from DAGonStar' > hello.txt")
workflow.add_task(hello)
workflow.make_dependencies()
workflow.run()
print(Path(hello.get_scratch_dir(), 'hello.txt').read_text())

## Lesson 02 — Add explicit dependencies

Use `add_dependency_to` for a conceptual or procedural ordering that is not inferred from an input file. Here the intended order is `prepare → simulate → summarize`.

In [ ]:
workdir = workspace('lesson02')
workflow = Workflow('Lesson02', config=local_config(workdir))
prepare = DagonTask(TaskType.BATCH, 'prepare', 'echo prepared > prepared.txt')
simulate = DagonTask(TaskType.BATCH, 'simulate', 'echo simulated > simulated.txt')
summarize = DagonTask(TaskType.BATCH, 'summarize', 'echo summarized > summarized.txt')
for task in (prepare, simulate, summarize):
    workflow.add_task(task)
simulate.add_dependency_to(prepare)
summarize.add_dependency_to(simulate)
workflow.Validate_WF()
workflow.run()
print('explicit dependency chain completed')

## Lesson 03 — Use `workflow://` data dependencies

A `workflow:///task/path` reference declares an input produced by another task. `make_dependencies()` discovers the edge and stages the referenced file for the consumer.

In [ ]:
workdir = workspace('lesson03')
workflow = Workflow('Lesson03', config=local_config(workdir))
observations = DagonTask(TaskType.BATCH, 'observations', "mkdir -p data; printf 'time,value\n0,12.5\n1,13.1\n' > data/series.csv")
statistics = DagonTask(TaskType.BATCH, 'statistics', 'wc -l workflow:///observations/data/series.csv > line_count.txt')
workflow.add_task(observations)
workflow.add_task(statistics)
workflow.make_dependencies()
workflow.run()
print(Path(statistics.get_scratch_dir(), 'line_count.txt').read_text())

## Lesson 04 — Inspect scratch directories and generated launchers

Each task has a scratch directory. Inspect it after a local run to find inputs, outputs, and generated command artifacts. Use Google Drive rather than `/content` when results must persist after a Colab reset.

In [ ]:
workdir = workspace('lesson04')
workflow = Workflow('Lesson04', config=local_config(workdir))
inspect = DagonTask(TaskType.BATCH, 'inspect', 'pwd > working_directory.txt; echo scratch-ready > result.txt')
workflow.add_task(inspect)
workflow.run()
scratch = Path(inspect.get_scratch_dir())
print('task scratch:', scratch)
print([path.name for path in scratch.iterdir()])

## Lesson 05 — Detect and fix dependency cycles

A DAG cannot contain a cycle. The first cell deliberately constructs one and catches the validation error; the second builds the corrected graph.

In [ ]:
workdir = workspace('lesson05-invalid')
invalid = Workflow('Lesson05Invalid', config=local_config(workdir))
a = DagonTask(TaskType.BATCH, 'A', 'echo A')
b = DagonTask(TaskType.BATCH, 'B', 'echo B')
invalid.add_task(a); invalid.add_task(b)
b.add_dependency_to(a); a.add_dependency_to(b)
try:
    invalid.Validate_WF()
except Exception as error:
    print('Expected cycle error:', error)

In [ ]:
workdir = workspace('lesson05-fixed')
workflow = Workflow('Lesson05Fixed', config=local_config(workdir))
a = DagonTask(TaskType.BATCH, 'A', 'echo A')
b = DagonTask(TaskType.BATCH, 'B', 'echo B')
workflow.add_task(a); workflow.add_task(b)
b.add_dependency_to(a)
workflow.Validate_WF(); workflow.run()
print('fixed DAG completed')

## Lesson 06 — Use checkpoint files

Checkpoint metadata lets a later workflow resume completed tasks while the recorded scratch directory still exists. For Colab persistence, keep the checkpoint and scratch directory in mounted Drive.

In [ ]:
workdir = workspace('lesson06')
checkpoint = workdir / 'lesson_06_checkpoint.json'
workflow = Workflow('Lesson06', config=local_config(workdir), checkpoint_file=str(checkpoint))
slow = DagonTask(TaskType.BATCH, 'slow', 'echo computed > result.txt')
workflow.add_task(slow); workflow.make_dependencies()
workflow.run(resume_checkpoint_file=str(checkpoint))
print(checkpoint.read_text()[:500])

## Lesson 07 — Choose data staging modes

Local `workflow://` file staging is directly supported in Colab, as demonstrated in Lesson 03. SCP, Globus, and SKYCDS require a separately configured remote service and must not receive credentials from notebook source.

In [ ]:
# The local staging example is deliberately small and self-contained.
workdir = workspace('lesson07')
workflow = Workflow('Lesson07', config=local_config(workdir))
source = DagonTask(TaskType.BATCH, 'source', "printf 'staged locally\n' > input.txt")
consumer = DagonTask(TaskType.BATCH, 'consumer', 'cat workflow:///source/input.txt > copied.txt')
workflow.add_task(source); workflow.add_task(consumer)
workflow.make_dependencies(); workflow.run()
print(Path(consumer.get_scratch_dir(), 'copied.txt').read_text())

## Lesson 08 — Run Docker-backed tasks

**Not supported in hosted Colab.** DAGonStar Docker tasks require a Docker daemon. Run this lesson on a workstation or remote VM where Docker is installed and the current user can access it. Colab can be used as a notebook client only when a remote Docker-capable backend is configured.

## Lesson 09 — Prepare remote and Slurm workflows

**Client-only in Colab.** A Colab runtime is neither an SSH target nor a Slurm cluster. You can create a workflow that submits to a reachable remote host after securely providing short-lived credentials and validating SSH connectivity. Do not paste private keys into this notebook. The following cell only performs safe local construction.

In [ ]:
from dagon.task import DagonTask, TaskType
task = DagonTask(TaskType.SLURM, 'check', 'echo check', partition='debug', ntasks=1, working_dir='scratch')
print(type(task).__name__)
print(task.generate_command('launcher.sh'))

## Lesson 10 — Compose meta-workflows

`DAG_TPS` coordinates multiple workflows and supports cross-workflow `workflow://OtherWorkflow/task/path` references. This local construction cell shows the graph; production use should be validated with site-specific examples.

In [ ]:
from dagon.dag_tps import DAG_TPS
workdir = workspace('lesson10')
observations = Workflow('Observations', config=local_config(workdir))
model = Workflow('Model', config=local_config(workdir))
collect = DagonTask(TaskType.BATCH, 'collect', "mkdir -p out; echo forcing > out/forcing.txt")
run_model = DagonTask(TaskType.BATCH, 'run_model', 'cat workflow://Observations/collect/out/forcing.txt > model_output.txt')
observations.add_task(collect); model.add_task(run_model)
meta = DAG_TPS('Lesson10', config=local_config(workdir)); meta.add_workflow(observations); meta.add_workflow(model)
meta.make_dependencies()
print(meta.as_json())

## Lesson 11 — Launch a workflow asynchronously

`launch()` runs a workflow in a background thread; `wait()` joins it. Keep a hosted Colab tab connected while work runs. For durable long jobs, submit to a remote backend instead.

In [ ]:
workdir = workspace('lesson11')
workflow = Workflow('Lesson11', config=local_config(workdir))
workflow.add_task(DagonTask(TaskType.BATCH, 'hello', 'echo hello'))
workflow.on_task_start += lambda task: print('starting', task.name)
workflow.on_task_end += lambda task: print('ended', task.name, task.status.name)
workflow.on_workflow_end += lambda wf: print('workflow complete')
workflow.launch()
print('The caller is free to do other work.')
workflow.wait()

## Lesson 12 — Run an LLM task locally

The repository example starts a local OpenAI-compatible mock provider, so it requires no credentials and runs in Colab.

In [ ]:
!python3 examples/tutorial/lesson_12_llm_tasks.py

## Lesson 13 — Execute a web task locally

The repository example starts a local HTTP service that is used only inside the runtime. It is not publicly exposed. Keep real authentication values in environment-backed secrets rather than notebook cells.

In [ ]:
!python3 examples/tutorial/lesson_13_web_tasks.py

## Lesson 14 — Execute native Python tasks

Native tasks call an importable module-level Python function while retaining DAGonStar input staging and declared output handling. The function module must be importable to the task runner. This cell creates a small temporary module and runs a native workflow.

In [ ]:
native_dir = workspace('lesson14')
(native_dir / 'native_functions.py').write_text(
    "def transform(input_file, scale, output_file):\n"
    "    values = [float(line.strip()) for line in open(input_file) if line.strip()]\n"
    "    with open(output_file, 'w') as destination:\n"
    "        for value in values: destination.write(f'{value * scale}\\n')\n"
    "    return {'count': len(values), 'scale': scale}\n"
)
import sys
sys.path.insert(0, str(native_dir))
workflow = Workflow('Lesson14', config=local_config(native_dir))
produce = DagonTask(TaskType.BATCH, 'produce', "mkdir -p data; printf '2\n4\n' > data/input.txt")
transform = DagonTask(TaskType.NATIVE, 'transform', 'native_functions:transform', inputs={'input_file': 'workflow:///produce/data/input.txt', 'scale': 1.5}, outputs={'output_file': 'scaled.txt'})
consume = DagonTask(TaskType.BATCH, 'consume', 'cat workflow:///transform/outputs/scaled.txt')
for task in (produce, transform, consume): workflow.add_task(task)
workflow.run()
print(Path(transform.working_dir, 'outputs', 'scaled.txt').read_text())

## Lesson 15 — Record FAIR metadata and provenance

FAIR recording is opt-in and local. This example declares a produced artifact and a derived artifact, preserves the `workflow://` dependency, and writes machine-readable metadata beside the temporary scratch directory. It requires no credentials or external repository.

In [ ]:
from dagon.fair import AccessPolicy, Agent, Artifact, FairProfile

fair_workdir = workspace('lesson15')
workflow = Workflow('Lesson15', config=local_config(fair_workdir))
workflow.enable_fair(FairProfile(
    title='A local FAIR workflow',
    description='Creates and copies a small research message.',
    creators=[Agent(name='DAGonStar tutorial')],
    license='Apache-2.0',
    keywords=['FAIR', 'provenance', 'tutorial'],
    access_policy=AccessPolicy(metadata='public', data='local'),
))
produce = DagonTask(TaskType.BATCH, 'produce', 'mkdir -p data; echo reproducible > data/message.txt').declare_outputs(
    Artifact('data/message.txt', media_type='text/plain', license='Apache-2.0')
).annotate(description='Create the primary local artifact.')
copy = DagonTask(TaskType.BATCH, 'copy', 'cat workflow:///produce/data/message.txt > copied.txt').declare_inputs(
    Artifact('workflow:///produce/data/message.txt')
).declare_outputs(Artifact('copied.txt', media_type='text/plain', license='Apache-2.0')).annotate(
    description='Create a derived artifact from the producer output.'
)
workflow.add_task(produce); workflow.add_task(copy)
workflow.run()
fair_dir = next((fair_workdir / '.dagon' / 'fair' / 'Lesson15').iterdir())
print('FAIR exports:', sorted(path.name for path in fair_dir.iterdir()))
print('Recorded tasks:', list(__import__('json').loads((fair_dir / 'run.json').read_text())['tasks']))
print('Derived output:', (Path(copy.working_dir) / 'copied.txt').read_text().strip())


## Next steps

For the complete prose, verification notes, and backend-specific guidance, see the corresponding files in [`docs/tutorial/`](https://github.com/DagOnStar/dagonstar/tree/master/docs/tutorial). Mount Google Drive before selecting persistent scratch or checkpoint locations. Keep generated notebooks free of credentials, and use a durable remote backend for long-running production workflows.